# 04 - Paper parity: where every source figure stands

**What this notebook is for.** The replication contract: every figure and table from the two
source papers is either replicated, substituted with the difference documented, queued with
its cost, or declared out of scope with the reason. Canonical inventory:
`notes/plot_parity.md` (rendered below); this notebook also carries the one result that lives
nowhere else, the logit-lens split verdict.

**Key concepts.**
- *Logit lens*: projecting an internal direction through the model's output vocabulary matrix
  to see which tokens it up-weights.
- *Substitution*: same analysis, different tool (for example t-SNE for UMAP), always with the
  difference stated.

**Index.**
1. The inventory (both papers)
2. Logit lens (paper Table 1): instruct fails, base partially reproduces

## 1. The inventory

### Anthropic paper

| Item | What it shows | Status | Where / why |
|---|---|---|---|
| Figure 1 | Top-activating dataset snippets per emotion vector, external corpora | QUEUED (scaled down) | Needs a corpus sweep (LMSYS/Pile samples) with per-token projection; ~half a pod day; not gate-critical |
| Table 1 | Logit-lens top/bottom tokens per emotion vector | SPLIT: base partial, instruct negative | Section 2 below: base vectors at layer 33 show affective token neighborhoods for about half the emotions; instruct vectors show none at layers 33 or 57. Final-norm scaling applied, softcapping ignored |
| Figure 2 | Probe x scenario cosine matrix, strong diagonal | DONE | notebooks/03, sections 1 and 3 (dual-model); our diagonals are weaker, which is a finding (TREE Q1.H2) |
| Table 2 | The 12 implicit-emotion scenarios | DONE | Used verbatim, src/emotion_vectors/probe_prompts.py |
| Figure 3 | Numerical-intensity template curves | DONE | notebooks/03, section 2 (dual-model): instruct tracks 11/11 registered directions, base 7/11 |
| Figure 4 | Activity-preference Elo + steering shifts | IN PROGRESS (Elo half) | TREE Q1.H3.E1 collecting on the pod: paper's preference prompt over a self-authored 64-activity list (the paper's list is unpublished; ours follows its 8 named categories, src/emotion_vectors/activities.py, committed before scoring). Steering half (E2) gated on E1 |
| Figure 5 | Pairwise cosine similarity, clustered | DONE | notebooks/02, section 3 |
| Figure 6 | UMAP of k-means emotion clusters | SUB, DONE | notebooks/archive/09: t-SNE embedding instead of UMAP (dependency), identical k-means k=10; clusters interpretable (joy/hope family, calm/content family), matching the paper's qualitative result |
| Figure 7 | PC1/PC2 loading bars per emotion | DONE | notebooks/02, section 5: each model in its own valence-best/arousal-best component plane |
| Figure 8 | PC1/PC2 vs human valence/arousal ratings | SUB | We correlate against the NRC VAD lexicon (the replication's instrument), not Russell's 45-emotion ratings; documented in TREE Q1.H1.C1 |
| Figure 9 | Representational similarity across layers | DONE | notebooks/02, section 4 |
| Neutral-PC projection (methods) | Confound removal before probe use | DONE (late) | Missing from the reference code and our pipeline until 2026-07-21; E7 implements it |
| Appendix: token-level activation localization | Vectors activate on emotion-relevant story spans | QUEUED | Requires per-token projection; shared infrastructure with Q3 |
| Overview panels: reward-hacking steering | Steering shifts misalignment rates | OUT | Production alignment evals and steering infra; not reproducible here |

### Open replication (sinievanderben/emotion_experiment)

| Item | What it shows | Status | Where / why |
|---|---|---|---|
| fig1_cosine_similarity | Contrast-vector cosine heatmap | DONE | notebooks/02, section 3 |
| fig2_pca | PCA scatter + valence/arousal panels | DONE | notebooks/02, sections 1-2 |
| fig3_umap | UMAP colored by k-means cluster | SUB, DONE | Same t-SNE substitution as the paper's Figure 6; notebooks/archive/09 |
| fig_valence/arousal_trajectory | PC-correlation across layers, two models | DONE + extended | notebooks/02 (base); our base-vs-instruct comparison (results/emotion_geometry_correlations*.json) is the same plot family with a new finding (valence demotion, TREE Q1.H1.C2) |
| fig_cka (centered kernel alignment) | Cross-layer representation similarity | SUB | We use representational similarity analysis (correlation of pairwise-cosine structures) instead of CKA; same question, different similarity index; notebooks/02 section 4 |
| analyze_story_conditions | Same model, vectors from different story corpora | DONE | Our E5 comparison: 4B-corpus vs self-generated probes (notebooks/archive/07) |
| visualize_token_activations | Per-token projection along a sentence | QUEUED | Becomes Q3's core infrastructure (per-token trajectories) |

## Maintenance

Update this table whenever a QUEUED item lands or a new figure appears in
either source. The notebook restyle (plotly, skimmable cells) references this
inventory so each notebook states which paper figure it corresponds to.


## 2. Logit lens (paper Table 1): instruct fails, base partially reproduces

In [1]:
# this cell renders the logit-lens token tables for both models: instruct (fails) then base (partial)
import json
from pathlib import Path

import plotly.graph_objects as go

ROOT = Path("..")
STRONG_BASE = [
    "happy",
    "proud",
    "desperate",
    "angry",
    "guilty",
]  # affective neighborhoods, judged by eye


def lens_table(result_path: Path, model_label: str) -> None:
    lens = json.loads(result_path.read_text())
    rows = [(e, ", ".join(t["up"]), ", ".join(t["down"])) for e, t in lens["table"].items()]
    fig = go.Figure(
        go.Table(
            header=dict(
                values=["emotion", "top up-weighted tokens", "top down-weighted tokens"],
                align="left",
            ),
            cells=dict(values=list(zip(*rows)), align="left", height=26),
        )
    )
    fig.update_layout(
        title=f"Logit lens, {model_label}, layer {lens['layer']} ({lens['note']})",
        height=460,
        margin=dict(t=50, b=10),
    )
    fig.show()


lens_table(ROOT / "results/logit_lens_it_L57.json", "gemma-4-31b-it")
lens_table(ROOT / "results/logit_lens_base_L33.json", "gemma-4-31b (base)")
print(
    "verdict, gemma-4-31b-it: no emotion-word neighborhoods at layer 33 or 57; Table 1 does not reproduce"
)
print(
    f"verdict, gemma-4-31b (base): affective neighborhoods for {len(STRONG_BASE)}/12 at layer 33 ({', '.join(STRONG_BASE)}); valence-consistent down-lists for most others"
)

verdict, gemma-4-31b-it: no emotion-word neighborhoods at layer 33 or 57; Table 1 does not reproduce
verdict, gemma-4-31b (base): affective neighborhoods for 5/12 at layer 33 (happy, proud, desperate, angry, guilty); valence-consistent down-lists for most others


<details><summary><b>How to read these tables</b></summary>

The paper's Table 1 shows each emotion vector up-weighting related words (sad toward grief, tears). Both tables apply the final-normalization scaling (Gemma's 1+w convention); the documented simplification is that logit softcapping is ignored.

- **Instruct (top table)**: unrelated fragments everywhere, at both tested layers (33 shown in `results/logit_lens_it_L33_normed.json`, 57 above). A robust negative.
- **Base (bottom table)**: clear affective neighborhoods for about half the emotions at layer 33 (happy toward delightful/wonderful, angry toward vicious/angrily, desperate toward misery/wretched, guilty toward conceal/incriminating), and valence-consistent down-lists for most others (sad down-weights charming/fabulous, calm down-weights brutal/vicious). Layer 57 (`results/logit_lens_base_L57.json`) is similar but noisier. A partial positive, judged qualitatively, no registered quantitative bar.

The split matters: the same extraction pipeline yields vocabulary-aligned directions on the base model and junk on the instruct model. This is the third independent signature (with the probe-battery failure and the valence demotion, TREE Q1.H1.C2) that instruction tuning buries the affect representation under non-affective structure.

</details>